# When AI Speaks, Markets Listen
## A3: Data Collection Notebook

**Purpose:** Collect, prepare, and document the datasets for the final research project.  
This notebook creates the daily panel dataset, event-level dataset, daily event-window dataset, and news datasets.

**Events:**
- GPT-4: March 15, 2023
- Gemini: December 6, 2023
- DeepSeek-R1: January 27, 2025

**Sample:** 80 stocks × 3 events = 240 planned firm-event observations  
The final event-level dataset contains 239 valid firm-event observations because one stock-event pair does not have enough pre-event observations for market model estimation.

**Data Collection Methods:**
1. REST API — yfinance: U.S. stock data and selected English news
2. CSMAR CSV — Chinese A-share daily stock data
3. Benchmark data — S&P 500 and CSI 300
4. Hidden API — East Money Chinese news, identified through Chrome DevTools
5. HTML Scraping — Reuters English news, using BeautifulSoup

---
## Before Running

- Run all cells in order from Step 0 to Step 9.
- The notebook assumes the GitHub repository has the required `data/`, `code/`, and `docs/` folders.
- Final cleaned datasets are saved in the `data/` folder.
- No API key is required.
- Raw CSMAR files are not included in the public repository because of database access restrictions.

---
## Step 0: Install Libraries

In [2]:
!pip install yfinance akshare pysentiment2 beautifulsoup4 requests pandas numpy -q

In [3]:
import os, time, json, warnings, requests
import numpy as np
import pandas as pd
import yfinance as yf
import akshare as ak
from bs4 import BeautifulSoup

warnings.filterwarnings('ignore')

# Create output folder
os.makedirs('data', exist_ok=True)

# ── Constants ──────────────────────────────────────────
START_DATE    = '2022-06-01'
END_DATE      = '2025-06-30'
START_DATE_CN = '20220601'   # AKShare uses YYYYMMDD format
END_DATE_CN   = '20250630'

GPT4_DATE = '2023-03-15'
DS_DATE   = '2025-01-27'

print('✅ Setup complete. data/ folder ready.')

✅ Setup complete. data/ folder ready.


---
## Step 1: Define Stock Universe

In [28]:
# U.S. Semiconductor — S&P 500 GICS 4530 (top 15 by AI revenue exposure)
US_SEMI = [
    "ADI", "AMAT", "AMD", "AVGO", "INTC",
    "KLAC", "LRCX", "MPWR", "MRVL", "MU",
    "NVDA", "ON", "QCOM", "TSM", "TXN",
    "ASML", "NXPI", "STM", "MCHP", "SWKS"
]

US_SOFT = [
    "ADBE", "CFLT", "CRM", "DDOG", "GOOGL",
    "INTU", "MDB", "META", "MSFT", "NOW",
    "ORCL", "PLTR", "SNOW", "WDAY", "ZS",
    "TEAM", "NET", "CRWD", "APP", "ESTC"
]

CN_SEMI = [
    "688008", "688009", "688012", "688036", "688041",
    "688047", "688065", "688099", "688256", "688271",
    "688396", "688469", "688521", "688598", "688981",
    "603986", "600584", "002371", "300782", "688072"
]

CN_SOFT = [
    "002230", "002410", "002908", "300024", "300058",
    "300229", "300308", "300418", "300450", "300866",
    "601360", "688111", "688188", "688579", "688787",
    "600570", "600588", "300454", "300496", "688327"
]

print(f'U.S. Semi:      {len(US_SEMI)} stocks')
print(f'U.S. Software:  {len(US_SOFT)} stocks')
print(f'China Semi:     {len(CN_SEMI)} stocks')
print(f'China Software: {len(CN_SOFT)} stocks')
print(f'Total:          {len(US_SEMI)+len(US_SOFT)+len(CN_SEMI)+len(CN_SOFT)} stocks')

U.S. Semi:      20 stocks
U.S. Software:  20 stocks
China Semi:     20 stocks
China Software: 20 stocks
Total:          80 stocks


---
## Step 2: U.S. Stock Data — yfinance REST API
✅ **WiFi or any connection. VPN is fine.**

Collects: daily close price, log return, volume, turnover rate, market cap, beta.

Saves to: `data/us_stock_data.csv`

In [38]:
import pandas as pd
import numpy as np
import yfinance as yf
import os
import time

DATA_DIR = "../data"
START_DATE = "2022-06-01"
END_DATE = "2026-04-30"

def fetch_us_stock(ticker, industry):
    try:
        df = yf.download(
            ticker,
            start=START_DATE,
            end=END_DATE,
            auto_adjust=True,
            progress=False
        )

        if df.empty:
            print(f"Empty US data: {ticker}")
            return None

        df = df.reset_index()

        if isinstance(df.columns, pd.MultiIndex):
            df.columns = [c[0] for c in df.columns]

        df = df.rename(columns={
            "Date": "date",
            "Close": "close",
            "Volume": "volume"
        })

        df["date"] = pd.to_datetime(df["date"])
        df["close"] = pd.to_numeric(df["close"], errors="coerce")
        df["volume"] = pd.to_numeric(df["volume"], errors="coerce")
        df["log_ret"] = np.log(df["close"] / df["close"].shift(1))

        df["ticker"] = ticker
        df["market"] = "US"
        df["industry"] = industry

        try:
            info_fast = yf.Ticker(ticker).fast_info
            df["marketcap"] = info_fast.get("market_cap", np.nan)
        except Exception:
            df["marketcap"] = np.nan

        try:
            info = yf.Ticker(ticker).info
            df["beta"] = info.get("beta", np.nan)
        except Exception:
            df["beta"] = np.nan

        df["turnover"] = df["volume"] / df["marketcap"]

        return df[
            [
                "date", "ticker", "market", "industry",
                "close", "log_ret", "volume", "turnover",
                "marketcap", "beta"
            ]
        ]

    except Exception as e:
        print(f"Failed {ticker}: {e}")
        return None


us_frames = []

for ticker in US_SEMI:
    print("US Semi:", ticker)
    df = fetch_us_stock(ticker, "Semi")
    if df is not None:
        us_frames.append(df)
    time.sleep(0.3)

for ticker in US_SOFT:
    print("US Software:", ticker)
    df = fetch_us_stock(ticker, "Software")
    if df is not None:
        us_frames.append(df)
    time.sleep(0.3)

us_stock_data = pd.concat(us_frames, ignore_index=True)
us_stock_data.to_csv(os.path.join(DATA_DIR, "us_stock_data.csv"), index=False)

print("\nSaved updated us_stock_data.csv")
print(us_stock_data.shape)
print(us_stock_data.groupby(["market", "industry"])["ticker"].nunique())
print("US Software tickers:")
print(sorted(us_stock_data[us_stock_data["industry"] == "Software"]["ticker"].unique()))

US Semi: ADI
US Semi: AMAT
US Semi: AMD
US Semi: AVGO
US Semi: INTC
US Semi: KLAC
US Semi: LRCX
US Semi: MPWR
US Semi: MRVL
US Semi: MU
US Semi: NVDA
US Semi: ON
US Semi: QCOM
US Semi: TSM
US Semi: TXN
US Semi: ASML
US Semi: NXPI
US Semi: STM
US Semi: MCHP
US Semi: SWKS
US Software: ADBE
US Software: CFLT
US Software: CRM
US Software: DDOG
US Software: GOOGL
US Software: INTU
US Software: MDB
US Software: META
US Software: MSFT
US Software: NOW
US Software: ORCL
US Software: PLTR
US Software: SNOW
US Software: WDAY
US Software: ZS
US Software: TEAM
US Software: NET
US Software: CRWD
US Software: APP
US Software: ESTC

Saved updated us_stock_data.csv
(39210, 10)
market  industry
US      Semi        20
        Software    20
Name: ticker, dtype: int64
US Software tickers:
['ADBE', 'APP', 'CFLT', 'CRM', 'CRWD', 'DDOG', 'ESTC', 'GOOGL', 'INTU', 'MDB', 'META', 'MSFT', 'NET', 'NOW', 'ORCL', 'PLTR', 'SNOW', 'TEAM', 'WDAY', 'ZS']


---
## Step 3: China A-Share Data — CSMAR CSV

Chinese A-share data were downloaded from the CSMAR database as CSV/Excel files and then cleaned in Python.

AKShare and TongHuaShun-related interfaces were tested earlier, but batch collection was unstable because repeated requests likely triggered anti-scraping restrictions. Therefore, the final Chinese stock dataset uses CSMAR data for better stability and reproducibility.

Collects:
- daily close price
- daily stock return
- turnover rate
- ticker
- trading date
- industry classification

Saves to: `data/cn_stock_data.csv`

---
## Step 4: Market Benchmarks — S&P 500 and CSI 300

✅ **U.S. benchmark (S&P 500): any connection**

⚠️ **China benchmark (CSI 300): mobile hotspot recommended**

Saves to: `data/sp500_benchmark.csv`, `data/csi300_benchmark.csv`

In [8]:
# ── S&P 500 (yfinance) ──────────────────────────────────
print('Fetching S&P 500 benchmark...')
try:
    raw = yf.download(
        '^GSPC', start=START_DATE, end=END_DATE, progress=False
    )
    sp500 = pd.DataFrame({
        'date':     pd.to_datetime(raw.index).tz_localize(None),
        'close':    raw['Close'].values.flatten(),
    })
    sp500['log_ret'] = np.log(sp500['close'] / sp500['close'].shift(1))
    sp500['index']   = 'SP500'
    sp500 = sp500.dropna()
    sp500.to_csv('data/sp500_benchmark.csv', index=False)
    print(f'✅ Saved: data/sp500_benchmark.csv ({len(sp500)} days)')
except Exception as e:
    print(f'❌ S&P 500 error: {e}')


Fetching S&P 500 benchmark...
✅ Saved: data/sp500_benchmark.csv (770 days)


In [9]:
# ── CSI 300 (AKShare) ────────────────────────────────────
print('\nFetching CSI 300 benchmark...')
print('⚠️  Make sure mobile hotspot is ON for this step')
try:
    raw_cn = ak.stock_zh_index_daily(symbol='sh000300')
    csi300 = raw_cn[['date','close']].copy()
    csi300['log_ret'] = np.log(
        csi300['close'] / csi300['close'].shift(1)
    )
    csi300['index'] = 'CSI300'
    csi300['date']  = pd.to_datetime(csi300['date'])
    csi300 = csi300.dropna()

    # Filter to study period
    csi300 = csi300[
        (csi300['date'] >= START_DATE) &
        (csi300['date'] <= END_DATE)
    ]
    csi300.to_csv('data/csi300_benchmark.csv', index=False)
    print(f'✅ Saved: data/csi300_benchmark.csv ({len(csi300)} days)')
except Exception as e:
    print(f'❌ CSI 300 error: {e}')
    print('   → Try switching to mobile hotspot')


Fetching CSI 300 benchmark...
⚠️  Make sure mobile hotspot is ON for this step
✅ Saved: data/csi300_benchmark.csv (747 days)


## Step 5: English News Collection — yfinance.news + Reuters HTML Scraping

**yfinance.news:** Built-in news API for selected U.S. AI-related stock tickers.  
**Reuters:** Static HTML page scraped using BeautifulSoup.  

This step collects English news headlines related to AI stocks and LLM events. The news data are used for event context and narrative interpretation.

✅ **Any connection works for this step.**

Saves to: `data/en_news_raw.csv`

In [34]:
import pandas as pd
import yfinance as yf
import requests
import time
from bs4 import BeautifulSoup

# ------------------------------------------------------------
# yfinance news
# ------------------------------------------------------------

def fetch_yf_news(tickers):
    """
    Collect English news headlines through yfinance's built-in news field.
    This version uses the updated yfinance structure where news information
    is stored inside the 'content' field.
    """
    records = []

    for ticker in tickers:
        try:
            news = yf.Ticker(ticker).news

            for n in news:
                content = n.get("content", {})
                title = content.get("title", "")
                publisher = content.get("provider", {}).get("displayName", "")
                date = content.get("pubDate", "")

                if title:
                    records.append({
                        "ticker": ticker,
                        "title": title,
                        "publisher": publisher,
                        "date": date,
                        "source": "yfinance",
                        "language": "en"
                    })

            time.sleep(0.3)

        except Exception as e:
            print(f"  ⚠️ {ticker}: {e}")

    return pd.DataFrame(records)


# ------------------------------------------------------------
# Reuters HTML scraping
# ------------------------------------------------------------

def fetch_reuters_news(query, pages=2):
    """
    Scrape Reuters search result headlines using BeautifulSoup.
    Method: requests.get -> BeautifulSoup -> parse headline tags.
    """
    records = []

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        )
    }

    for page in range(pages):
        try:
            url = (
                "https://www.reuters.com/search/news"
                f"?query={query}&offset={page * 20}"
            )

            r = requests.get(url, headers=headers, timeout=10)
            soup = BeautifulSoup(r.text, "html.parser")

            found = (
                soup.find_all("h3", class_="search-result-title") or
                soup.find_all("a", attrs={"data-testid": "Heading"}) or
                soup.find_all("h3")
            )

            count = 0

            for tag in found:
                title = tag.get_text(strip=True)

                if title and len(title) > 15:
                    records.append({
                        "ticker": None,
                        "title": title,
                        "publisher": "Reuters",
                        "date": None,
                        "source": "reuters_scrape",
                        "language": "en",
                        "query": query
                    })
                    count += 1

            print(f"Reuters page {page + 1}, query={query}: {count} headlines")
            time.sleep(2)

        except Exception as e:
            print(f"Reuters page {page + 1}, query={query}, error: {e}")

    return pd.DataFrame(records)


# ------------------------------------------------------------
# Run English news collection
# ------------------------------------------------------------

print("Collecting English news...")
print("=" * 50)

key_tickers = ["NVDA", "AMD", "MSFT", "GOOGL", "META", "CRM", "AVGO"]

print("\n[yfinance News]")
yf_news = fetch_yf_news(key_tickers)
print(f"Collected from yfinance: {len(yf_news)} articles")

print("\n[Reuters HTML Scraping]")
reuters_ds = fetch_reuters_news("DeepSeek+AI+stock", pages=2)
reuters_gpt = fetch_reuters_news("GPT-4+stock+market", pages=2)
reuters_gemini = fetch_reuters_news("Gemini+AI+stock", pages=2)

en_news = pd.concat(
    [yf_news, reuters_ds, reuters_gpt, reuters_gemini],
    ignore_index=True
)

DATA_DIR = "../data" if os.path.exists("../data") else os.path.expanduser("~/Desktop")
output_path = os.path.join(DATA_DIR, "en_news_raw.csv")

en_news.to_csv(output_path, index=False)

print(f"\nSaved: {output_path}")
print(f"Total English news articles: {len(en_news)}")
print(en_news.head())



[yfinance News]
Collected from yfinance: 70 articles

[Reuters HTML Scraping]
Reuters page 1, query=DeepSeek+AI+stock: 0 headlines
Reuters page 2, query=DeepSeek+AI+stock: 0 headlines
Reuters page 1, query=GPT-4+stock+market: 0 headlines
Reuters page 2, query=GPT-4+stock+market: 0 headlines
Reuters page 1, query=Gemini+AI+stock: 0 headlines
Reuters page 2, query=Gemini+AI+stock: 0 headlines

Saved: ../data/en_news_raw.csv
Total English news articles: 70
  ticker                                              title  \
0   NVDA  'Nobody's doing AI better than Palantir': Earn...   
1   NVDA  Nvidia CEO Jensen Huang says company now has z...   
2   NVDA  Palantir Stock: With a Fresh Earnings Report S...   
3   NVDA  As workers worry about AI, Nvidia’s Jensen Hua...   
4   NVDA              Why Upstart Stock Jumped 23% in April   

             publisher                  date    source language  
0  Yahoo Finance Video  2026-05-04T21:20:21Z  yfinance       en  
1        Yahoo Finance  2026-0

---
## Step 6: Chinese News — East Money Hidden API

**Method:** Hidden API discovered via Chrome DevTools Network panel

**How to discover it yourself:**
1. Open www.eastmoney.com
2. Press F12 → Network tab → Filter: XHR
3. Search for 'DeepSeek' in the search bar
4. Find the request to `search-api-web.eastmoney.com`
5. Copy the URL and parameters

**Response format:** JSONP (needs jQuery wrapper removed before parsing)

✅ **Any connection works for this step.**

Saves to: `data/cn_news_raw.csv`

In [11]:
def fetch_eastmoney_news(keyword, pages=3):
    """
    Collect Chinese financial news via East Money hidden API.
    Endpoint discovered via Chrome DevTools → Network → XHR filter.
    Response is JSONP: jQuery({...}) — must strip wrapper before json.loads().
    """
    records = []
    headers = {
        'User-Agent': (
            'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
            'AppleWebKit/537.36 (KHTML, like Gecko) '
            'Chrome/120.0.0.0 Safari/537.36'
        ),
        'Referer': 'https://www.eastmoney.com/'
    }

    for page in range(1, pages + 1):
        try:
            # Hidden API endpoint
            url    = 'https://search-api-web.eastmoney.com/search/jsonp'
            params = {
                'keyword':   keyword,
                'type':      '14',   # 14 = news
                'pageindex': page,
                'pagesize':  20,
                'callback':  'jQuery'
            }

            r    = requests.get(
                url, params=params, headers=headers, timeout=10
            )
            text = r.text

            # Strip JSONP wrapper: jQuery({...}) → {...}
            if 'jQuery' in text and '(' in text:
                text = text[text.index('(') + 1: text.rindex(')')]
            data  = json.loads(text)
            items = data.get('data', {}).get('list', [])

            for item in items:
                records.append({
                    'keyword':   keyword,
                    'title':     item.get('title', ''),
                    'date':      item.get('datetime', ''),
                    'publisher': item.get('mediaName', ''),
                    'source':    'eastmoney_hidden_api',
                    'language':  'zh'
                })

            print(f'    Page {page}: {len(items)} articles')
            time.sleep(1.5)

        except Exception as e:
            print(f'    Page {page} error: {e}')

    return pd.DataFrame(records)


# ── Run collection ───────────────────────────────────────
print('Collecting Chinese news via East Money Hidden API...')
print('(Endpoint discovered via Chrome DevTools Network panel)')
print('=' * 50)

print('\nKeyword: DeepSeek')
cn_news_ds  = fetch_eastmoney_news('DeepSeek', pages=3)

print('\nKeyword: GPT-4')
cn_news_gpt = fetch_eastmoney_news('GPT-4', pages=3)

cn_news = pd.concat([cn_news_ds, cn_news_gpt], ignore_index=True)
cn_news.to_csv('data/cn_news_raw.csv', index=False)
print(f'\n✅ Saved: data/cn_news_raw.csv ({len(cn_news)} articles)')

(Endpoint discovered via Chrome DevTools Network panel)

Keyword: DeepSeek
    Page 1: 0 articles
    Page 2: 0 articles
    Page 3: 0 articles

Keyword: GPT-4
    Page 1: 0 articles
    Page 2: 0 articles
    Page 3: 0 articles

✅ Saved: data/cn_news_raw.csv (0 articles)


## Step 7: Load Final Daily Panel Dataset

This section loads the final cleaned daily panel dataset `data/master_data.csv`. The intermediate Chinese CSMAR files are not included in the public repository because of database access restrictions. The final cleaned Chinese observations are already included in `master_data.csv`.

In [43]:
import pandas as pd
import os

DATA_DIR = "../data"

master_data = pd.read_csv(os.path.join(DATA_DIR, "master_data.csv"))
master_data["date"] = pd.to_datetime(master_data["date"])

print("Loaded final master_data.csv")
print(master_data.shape)
print(master_data.groupby(["market", "industry"])["ticker"].nunique())
print(master_data.head())

Loaded final master_data.csv
(76814, 15)
market  industry
China   Semi        20
        Software    20
US      Semi        20
        Software    20
Name: ticker, dtype: int64
        date ticker market industry   close  log_ret  volume  turnover  \
0 2022-06-01   2371  China     Semi  272.46      NaN     NaN  1.307334   
1 2022-06-02   2371  China     Semi  288.71      NaN     NaN  2.124408   
2 2022-06-06   2371  China     Semi  291.50      NaN     NaN  1.645783   
3 2022-06-07   2371  China     Semi  278.18      NaN     NaN  1.480556   
4 2022-06-08   2371  China     Semi  279.70      NaN     NaN  1.289143   

   marketcap  beta       ret  ret_market  stock_ret  China  Software  
0        NaN   NaN  0.009111   -0.002041   0.009111      1         0  
1        NaN   NaN  0.059642    0.001564   0.059642      1         0  
2        NaN   NaN  0.009664    0.018537   0.009664      1         0  
3        NaN   NaN -0.045695    0.003126  -0.045695      1         0  
4        NaN   NaN  0.0

## Step 8: Event-Level Dataset Construction

This section constructs the event-study datasets based on `master_data.csv`. It defines three LLM release events: GPT-4, Gemini, and DeepSeek-R1. For each stock-event pair, it estimates the market model and calculates abnormal returns, CAR[-5,+10], PostCAR[+11,+30], TurnoverChange, and estimated beta.

Saves to:
- `data/event_level_data.csv`
- `data/daily_event_data.csv`

In [39]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import os

# ============================================================
# Step 6: Event-Level Dataset Construction
# Read master_data.csv from data folder
# Save event_level_data.csv and daily_event_data.csv to data folder
# ============================================================

DATA_DIR = "../data"
MASTER_PATH = os.path.join(DATA_DIR, "master_data.csv")

master_data = pd.read_csv(MASTER_PATH)
master_data["date"] = pd.to_datetime(master_data["date"])

if "stock_ret" not in master_data.columns:
    master_data["stock_ret"] = master_data["log_ret"].combine_first(master_data["ret"])

master_data = master_data.sort_values(["market", "ticker", "date"]).reset_index(drop=True)

print("Master data loaded:")
print(master_data.shape)
print(master_data.groupby(["market", "industry"])["ticker"].nunique())

# ------------------------------------------------------------
# Events
# ------------------------------------------------------------

events = {
    "GPT4": pd.Timestamp("2023-03-15"),
    "Gemini": pd.Timestamp("2023-12-06"),
    "DeepSeek_R1": pd.Timestamp("2025-01-27")
}

EST_START = -120
EST_END = -11
EVENT_START = -5
EVENT_END = 10
POST_START = 11
POST_END = 30


def assign_event_day(stock_df, event_date):
    stock_df = stock_df.sort_values("date").copy()
    dates = stock_df["date"].reset_index(drop=True)

    possible_event_days = dates[dates >= event_date]

    if possible_event_days.empty:
        return None

    actual_event_date = possible_event_days.iloc[0]
    event_position = dates[dates == actual_event_date].index[0]

    stock_df = stock_df.reset_index(drop=True)
    stock_df["event_day"] = stock_df.index - event_position
    stock_df["actual_event_date"] = actual_event_date

    return stock_df


event_rows = []
daily_event_frames = []

for event_name, event_date in events.items():
    print(f"\nProcessing event: {event_name}, event date = {event_date.date()}")

    for ticker, stock_df in master_data.groupby("ticker"):
        stock_df = stock_df.sort_values("date").copy()

        market = stock_df["market"].dropna().iloc[0]
        industry = stock_df["industry"].dropna().iloc[0]

        event_df = assign_event_day(stock_df, event_date)

        if event_df is None:
            print(f"Skipped {ticker} {event_name}: no trading date after event")
            continue

        event_df = event_df[
            (event_df["event_day"] >= EST_START) &
            (event_df["event_day"] <= POST_END)
        ].copy()

        valid_df = event_df.dropna(subset=["stock_ret", "ret_market"])

        est_df = valid_df[
            (valid_df["event_day"] >= EST_START) &
            (valid_df["event_day"] <= EST_END)
        ].copy()

        if len(est_df) < 30:
            print(f"Skipped {ticker} {event_name}: not enough estimation obs ({len(est_df)})")
            continue

        X = sm.add_constant(est_df["ret_market"])
        y = est_df["stock_ret"]

        try:
            model = sm.OLS(y, X).fit()
            alpha = model.params["const"]
            beta_estimated = model.params["ret_market"]
        except Exception as e:
            print(f"Skipped {ticker} {event_name}: regression failed: {e}")
            continue

        event_df["expected_ret"] = alpha + beta_estimated * event_df["ret_market"]
        event_df["abnormal_ret"] = event_df["stock_ret"] - event_df["expected_ret"]

        event_window = event_df[
            (event_df["event_day"] >= EVENT_START) &
            (event_df["event_day"] <= EVENT_END)
        ].dropna(subset=["abnormal_ret"])

        post_window = event_df[
            (event_df["event_day"] >= POST_START) &
            (event_df["event_day"] <= POST_END)
        ].dropna(subset=["abnormal_ret"])

        est_turnover = event_df[
            (event_df["event_day"] >= EST_START) &
            (event_df["event_day"] <= EST_END)
        ]["turnover"].mean()

        event_turnover = event_df[
            (event_df["event_day"] >= EVENT_START) &
            (event_df["event_day"] <= EVENT_END)
        ]["turnover"].mean()

        turnover_change = event_turnover - est_turnover

        if pd.notna(est_turnover) and est_turnover != 0:
            turnover_change_ratio = event_turnover / est_turnover - 1
        else:
            turnover_change_ratio = np.nan

        marketcap_values = stock_df["marketcap"].dropna()
        marketcap = marketcap_values.iloc[0] if len(marketcap_values) > 0 else np.nan

        daily_temp = event_df.copy()
        daily_temp["event"] = event_name
        daily_temp["ticker"] = ticker
        daily_temp["market"] = market
        daily_temp["industry"] = industry
        daily_event_frames.append(daily_temp)

        event_rows.append({
            "ticker": ticker,
            "market": market,
            "industry": industry,
            "event": event_name,
            "planned_event_date": event_date,
            "actual_event_date": event_df["actual_event_date"].iloc[0],
            "alpha": alpha,
            "beta_estimated": beta_estimated,
            "marketcap": marketcap,
            "CAR_m5_p10": event_window["abnormal_ret"].sum(),
            "PostCAR_p11_p30": post_window["abnormal_ret"].sum(),
            "Turnover_Estimation_Avg": est_turnover,
            "Turnover_Event_Avg": event_turnover,
            "TurnoverChange": turnover_change,
            "TurnoverChangeRatio": turnover_change_ratio,
            "event_window_obs": len(event_window),
            "post_window_obs": len(post_window),
            "estimation_obs": len(est_df),
            "China": 1 if market == "China" else 0,
            "Software": 1 if industry == "Software" else 0
        })


# ------------------------------------------------------------
# Save event-level dataset
# ------------------------------------------------------------

event_level_data = pd.DataFrame(event_rows)
event_level_data = event_level_data.sort_values(
    ["event", "market", "industry", "ticker"]
).reset_index(drop=True)

EVENT_LEVEL_PATH = os.path.join(DATA_DIR, "event_level_data.csv")
event_level_data.to_csv(EVENT_LEVEL_PATH, index=False)

print("\nSaved event-level dataset:")
print(EVENT_LEVEL_PATH)
print(event_level_data.shape)
print(event_level_data.groupby(["event", "market", "industry"]).size())


# ------------------------------------------------------------
# Save daily event dataset
# ------------------------------------------------------------

daily_event_data = pd.concat(daily_event_frames, ignore_index=True)

DAILY_EVENT_PATH = os.path.join(DATA_DIR, "daily_event_data.csv")
daily_event_data.to_csv(DAILY_EVENT_PATH, index=False)

print("\nSaved daily event dataset:")
print(DAILY_EVENT_PATH)
print(daily_event_data.shape)

print("\nMissing values in key event-level variables:")
print(event_level_data[
    [
        "CAR_m5_p10",
        "PostCAR_p11_p30",
        "TurnoverChange",
        "TurnoverChangeRatio",
        "marketcap",
        "beta_estimated"
    ]
].isna().sum())

Master data loaded:
(76814, 15)
market  industry
China   Semi        20
        Software    20
US      Semi        20
        Software    20
Name: ticker, dtype: int64

Processing event: GPT4, event date = 2023-03-15
Skipped 688469 GPT4: not enough estimation obs (0)

Processing event: Gemini, event date = 2023-12-06

Processing event: DeepSeek_R1, event date = 2025-01-27

Saved event-level dataset:
../data/event_level_data.csv
(239, 20)
event        market  industry
DeepSeek_R1  China   Semi        20
                     Software    20
             US      Semi        20
                     Software    20
GPT4         China   Semi        19
                     Software    20
             US      Semi        20
                     Software    20
Gemini       China   Semi        20
                     Software    20
             US      Semi        20
                     Software    20
dtype: int64

Saved daily event dataset:
../data/daily_event_data.csv
(36089, 20)

Missing value

## Step 9: Collection Summary and Final Checks

In [36]:
import pandas as pd
import os

DATA_DIR = "../data" if os.path.exists("../data") else os.path.expanduser("~/Desktop")

# Load final datasets
master_data = pd.read_csv(os.path.join(DATA_DIR, "master_data.csv"))
event_level_data = pd.read_csv(os.path.join(DATA_DIR, "event_level_data.csv"))
daily_event_data = pd.read_csv(os.path.join(DATA_DIR, "daily_event_data.csv"))

# Make sure date is datetime
master_data["date"] = pd.to_datetime(master_data["date"])

# Create unified stock return variable if needed
if "stock_ret" not in master_data.columns:
    master_data["stock_ret"] = master_data["log_ret"].combine_first(master_data["ret"])

# Sort dataset
master_data = master_data.sort_values(["market", "ticker", "date"])

print("Final Dataset Summary")
print("=" * 50)

print("\nmaster_data.csv")
print("Shape:", master_data.shape)
print(master_data.groupby(["market", "industry"])["ticker"].nunique())

print("\nevent_level_data.csv")
print("Shape:", event_level_data.shape)
print(event_level_data["event"].value_counts())
print(event_level_data.groupby(["event", "market", "industry"]).size())

print("\ndaily_event_data.csv")
print("Shape:", daily_event_data.shape)
print(daily_event_data["event"].value_counts())

print("\nMissing values in key master variables:")
print(master_data[["stock_ret", "ret_market", "turnover"]].isna().sum())

print("\nDescriptive statistics:")
desc_stats = master_data[["close", "stock_ret", "volume", "turnover", "ret_market"]].describe()
print(desc_stats)

print("\nStock return statistics by market and industry:")
group_stats = master_data.groupby(["market", "industry"])["stock_ret"].describe()
print(group_stats)

Final Dataset Summary

master_data.csv
Shape: (76814, 15)
market  industry
China   Semi        20
        Software    20
US      Semi        20
        Software    20
Name: ticker, dtype: int64

event_level_data.csv
Shape: (239, 20)
event
DeepSeek_R1    80
Gemini         80
GPT4           79
Name: count, dtype: int64
event        market  industry
DeepSeek_R1  China   Semi        20
                     Software    20
             US      Semi        20
                     Software    20
GPT4         China   Semi        19
                     Software    20
             US      Semi        20
                     Software    20
Gemini       China   Semi        20
                     Software    20
             US      Semi        20
                     Software    20
dtype: int64

daily_event_data.csv
Shape: (36089, 20)
event
Gemini         12080
DeepSeek_R1    12080
GPT4           11929
Name: count, dtype: int64

Missing values in key master variables:
stock_ret        40
ret_marke

## Variable Dictionary

| Variable | Type | Description | Example |
|---|---|---|---|
| `date` | Date | Trading date | `2023-03-15` |
| `ticker` | String | Stock ticker | `NVDA` |
| `market` | String | Market indicator: U.S. or China | `US` |
| `industry` | String | Industry group: semiconductor or software | `Semi` |
| `close` | Float | Daily closing price | `18.29` |
| `log_ret` | Float | Daily log return, mainly for U.S. stocks | `0.012` |
| `ret` | Float | Daily return, mainly for Chinese stocks | `0.015` |
| `stock_ret` | Float | Unified daily stock return variable, combining `log_ret` and `ret` | `0.012` |
| `volume` | Float | Daily trading volume | `544514000` |
| `turnover` | Float | Daily turnover rate | `1.31` |
| `marketcap` | Float | Firm market capitalization where available | `4.82e12` |
| `beta` | Float | Source-provided beta where available | `2.335` |
| `ret_market` | Float | Daily market benchmark return | `0.018` |
| `China` | Integer | Dummy variable equal to 1 for Chinese firms | `0` |
| `Software` | Integer | Dummy variable equal to 1 for software firms | `1` |
| `event` | String | LLM release event | `Gemini` |
| `planned_event_date` | Date | Original LLM release date used in the event study | `2023-12-06` |
| `actual_event_date` | Date | First available trading day on or after the planned event date | `2023-12-06` |
| `event_day` | Integer | Trading-day index relative to the event date | `0` |
| `alpha` | Float | Estimated intercept from the market model | `0.0001` |
| `beta_estimated` | Float | Estimated beta from the market model | `1.43` |
| `expected_ret` | Float | Expected return estimated from the market model | `0.006` |
| `abnormal_ret` | Float | Stock return minus expected return | `0.021` |
| `CAR_m5_p10` | Float | Cumulative abnormal return from event day -5 to +10 | `0.034` |
| `PostCAR_p11_p30` | Float | Cumulative abnormal return from event day +11 to +30 | `-0.012` |
| `Turnover_Estimation_Avg` | Float | Average turnover during the estimation window | `1.20` |
| `Turnover_Event_Avg` | Float | Average turnover during the event window | `1.62` |
| `TurnoverChange` | Float | Event-window average turnover minus estimation-window average turnover | `0.42` |
| `TurnoverChangeRatio` | Float | Percentage change in turnover from estimation window to event window | `0.18` |
| `event_window_obs` | Integer | Number of valid observations in the event window | `16` |
| `post_window_obs` | Integer | Number of valid observations in the post-event window | `20` |
| `estimation_obs` | Integer | Number of valid observations in the estimation window | `110` |

## Notes

### Sample Expansion

To address the concern about limited sample size, the sample was expanded from 60 firms to 80 firms. Each market-industry group now contains 20 firms:

- U.S. semiconductor firms: 20
- U.S. software firms: 20
- Chinese semiconductor firms: 20
- Chinese software firms: 20

The additional firms were selected based on three criteria: AI relevance, market capitalization or industry representativeness, and data availability.

### Event Expansion

The updated event-study design includes three LLM release events:

| Event | Event Date | Origin |
|---|---:|---|
| GPT-4 | 2023-03-15 | U.S. |
| Gemini | 2023-12-06 | U.S. |
| DeepSeek-R1 | 2025-01-27 | China |

Adding Gemini increases the planned event-level sample from 120 firm-event observations to 240.

### Final Dataset Size

The final datasets are:

| Dataset | Rows | Columns | Description |
|---|---:|---:|---|
| `master_data.csv` | 76,814 | 15 | Daily firm-date panel dataset |
| `event_level_data.csv` | 239 | 20 | Firm-event event-study dataset |
| `daily_event_data.csv` | 36,089 | 20 | Daily event-window abnormal return dataset |

The event-level dataset contains 239 observations instead of 240 because one firm-event does not have enough pre-event observations to estimate the market model.

### Known Data Quality Issues

- U.S. and Chinese markets follow different trading calendars, so event windows are constructed using trading days rather than calendar days.
- Some benchmark returns are missing because market holidays do not overlap across the U.S. and China.
- Turnover is missing for many U.S. observations because it depends on available market capitalization data.
- Chinese firm-level market capitalization is incomplete.
- The DeepSeek-R1 event date is close to the Chinese Spring Festival closure, which may affect the Chinese event window.